# Chapter 8 — Linking Ontologies to Data
### Notebook 1 · Mappings and materialisation

*Book reference: Sections 8.1–8.2*

A mapping says how rows become triples. It is a small artefact with one very sharp edge: the difference between an IRI and a literal decides whether your graph is connected or quietly in pieces.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch08_toolkit as ch8
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The source knows nothing about your ontology

A hospital database, of the kind §8.1 is written about. Note what it does *not* contain: any notion of a `Patient` class, a `Disorder`, or a category like 'cardiac'. Those live only in the ontology.

In [ ]:
print(ch8.SCHEMA_SQL)

In [ ]:
conn = ch8.build_database()
for table in ['ward', 'patient', 'diagnosis', 'code_lookup']:
    rows = conn.execute(f'SELECT * FROM {table}').fetchall()
    names = [d[0] for d in conn.execute(f'SELECT * FROM {table}').description]
    print(f'-- {table} --')
    print(pd.DataFrame(rows, columns=names).to_string(index=False))
    print()

## 2. The mappings

Two kinds, following R2RML:

* a **class map** turns each row of a table into an instance, with an IRI built from the primary key;
* a **property map** turns a pair of columns into a triple. Its object is either an **IRI** (when the column references another table) or a **literal** (when it holds a plain value).

In [ ]:
print(pd.DataFrame(ch8.mapping_report()).to_string(index=False))

In [ ]:
patient_map = ch8.PATIENT
print('class map :', patient_map.rdf_class.split('#')[-1])
print('  table   :', patient_map.table)
print('  IRI for row id=101 ->', patient_map.iri(101))

inward = ch8.MAPPINGS['properties'][0]
print('\nproperty map:', inward.predicate.split('#')[-1])
print('  ', f'{inward.table}.{inward.subject_column}',
      '->', f'{inward.table}.{inward.object_column}')
print('  object is an IRI:', inward.object_template is not None)

## 3. Materialising

Run every mapping, collect triples. This is the ETL answer: pay the transformation once.

In [ ]:
graph = ch8.materialise(conn)
print(f'{len(graph)} triples\n')
print(graph.serialize(format='turtle')[:900])

In [ ]:
from rdflib import URIRef
patient101 = URIRef('http://example.org/data/patient/101')
print('everything the graph knows about patient 101:')
for p, o in graph.predicate_objects(patient101):
    print(f'  {p.split("#")[-1].split("/")[-1]:14s} {o}')

## 4. The sharp edge: IRI or literal?

`patient.ward_id` is a foreign key. Mapped to an **IRI** it links the patient to the ward *resource*. Mapped to a **literal** it produces the number `1` — true, useless, and silent. Watch what breaks.

In [ ]:
import copy
broken = {'classes': ch8.MAPPINGS['classes'],
          'properties': [ch8.PropertyMap(p.predicate, p.table, p.subject_column,
                                         p.object_column, p.subject_template,
                                         None if p.predicate.endswith('inWard')
                                         else p.object_template)
                         for p in ch8.MAPPINGS['properties']]}
broken_graph = ch8.materialise(conn, broken)
print('triples, correct mapping:', len(graph))
print('triples, broken mapping :', len(broken_graph))
print('\nSame count. Nothing failed, nothing warned.')

In [ ]:
query = ch8.QUERIES['patients-in-cardiology']
good = ch8.answers_via_materialisation(conn, query, graph=graph)
bad = ch8.answers_via_materialisation(conn, query, mappings=broken)
print('patients in cardiology, correct mapping:', len(good))
print('patients in cardiology, broken mapping :', len(bad))
assert len(good) == 3 and len(bad) == 0
print('\nThe join through ward_id can never match, because one side is a\n'
      'resource and the other is the number 1. The query returns nothing and\n'
      'no component reports an error. This is the most common OBDA bug and it\n'
      'is invisible to every check except actually running a query.')

### Exercise 1.1 — Map a new column

The `patient` table has a `name` column. Add a property map producing `med:patientName` as a **literal**, materialise, and confirm the new triples appear.

> **Hint.** `object_template=None` makes the object a literal.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
extra = ch8.PropertyMap(ch8.MED + 'patientName', 'patient', 'id', 'name',
                        'http://example.org/data/patient/{}', None)
mappings = {'classes': ch8.MAPPINGS['classes'],
            'properties': ch8.MAPPINGS['properties'] + [extra]}
extended = ch8.materialise(conn, mappings)
print('triples before:', len(graph), '-> after:', len(extended))
assert len(extended) == len(graph) + 5
rows = extended.query('''PREFIX med: <http://example.org/med#>
    SELECT ?p ?n WHERE { ?p med:patientName ?n }''')
for row in sorted(str(r[1]) for r in rows):
    print('  ', row)
print('\nFive patients, five literals. A name is an attribute, not a reference,\n'
      'so a literal is right here -- the opposite call from ward_id.')

### Exercise 1.2 — Measure what materialisation costs

Materialisation trades storage for query speed. Grow the database and measure how the triple count scales against the row count.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
rows = []
for n_patients in [5, 50, 500]:
    data = {k: list(v) for k, v in ch8.SAMPLE_ROWS.items()}
    data['patient'] = [(100 + i, f'P{i}', (i % 3) + 1) for i in range(n_patients)]
    data['diagnosis'] = [(100 + i, ['I21', 'I50', 'J45'][i % 3])
                         for i in range(n_patients)]
    c = ch8.build_database(data)
    g = ch8.materialise(c)
    source_rows = sum(len(v) for v in data.values())
    rows.append({'patients': n_patients, 'source rows': source_rows,
                 'triples': len(g),
                 'triples per row': round(len(g) / source_rows, 2)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nRoughly linear, at about three triples per source row. That is the\n'
      'storage bill for materialisation -- and it must be paid again, in full,\n'
      'every time the source changes enough to matter.')